# Project - Fundamentals of AI & Data Science
Student ID: B00172277
Dataset: Dataset2_odd_students.csv


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay


In [ ]:
df = pd.read_csv('Dataset2_odd_students.csv')
df = df.dropna()
df.head()

In [ ]:
y = df.iloc[:,-1]
X = df.iloc[:,:-1]

cat_cols = X.select_dtypes(include=['object']).columns
encoder = OrdinalEncoder()
X[cat_cols] = encoder.fit_transform(X[cat_cols])

X = X - X.min()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


## Data Visualisation

In [ ]:
X.hist(figsize=(12,10))
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(X.corr(), cmap='coolwarm')
plt.show()

## Naïve Bayes

In [ ]:
nb = ComplementNB(alpha=0, force_alpha=True, norm=False)
nb.fit(X_train, y_train)
print('Baseline NB Accuracy:', accuracy_score(y_test, nb.predict(X_test)))


In [ ]:
param_grid = {
 'alpha':[0.01,0.1,0.5,1.0,2.0,10.0],
 'norm':[True,False]
}

grid = GridSearchCV(ComplementNB(), param_grid, cv=StratifiedKFold(10), scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_nb = grid.best_estimator_
print('Best NB Params:', grid.best_params_)
print('NB Accuracy:', accuracy_score(y_test, best_nb.predict(X_test)))

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_nb, X_test, y_test)
plt.title('NB Confusion Matrix')
plt.show()

print(classification_report(y_test, best_nb.predict(X_test)))

In [ ]:
scores = cross_val_score(best_nb, X, y, cv=5)
print('NB Cross-val Accuracy:', scores.mean())

## Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print('Baseline DT Accuracy:', accuracy_score(y_test, dt.predict(X_test)))

In [ ]:
param_grid_dt = {
 'criterion':['gini','entropy'],
 'max_depth':[5,10,15,20,None],
 'min_samples_split':[2,5,10,20],
 'min_samples_leaf':[1,2,5,10],
 'min_impurity_decrease':[0.0,0.01,0.05]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=5, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
print('Best DT Params:', grid_dt.best_params_)
print('DT Accuracy:', accuracy_score(y_test, best_dt.predict(X_test)))

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_dt, X_test, y_test)
plt.title('DT Confusion Matrix')
plt.show()

print(classification_report(y_test, best_dt.predict(X_test)))

## Model Comparison

In [ ]:
nb_acc = accuracy_score(y_test, best_nb.predict(X_test))
dt_acc = accuracy_score(y_test, best_dt.predict(X_test))

print('Naive Bayes:', nb_acc)
print('Decision Tree:', dt_acc)